In [2]:
import pandas as pd

df = pd.read_csv("dataset/new.csv", encoding="latin1")

print(df.head())
print(df.columns)


   doc ID                    title  \
0     1.0  Artificial intelligence   
1     NaN                      NaN   
2     NaN                      NaN   
3     NaN                      NaN   
4     NaN                      NaN   

                                                text  
0  Artificial intelligence (AI) is the capability...  
1  associated with human intelligence, such as le...  
2  perception, and decision-making. It is a field...  
3  and studies methods and software that enable m...  
4  use learning and intelligence to take actions ...  
Index(['doc ID', 'title', 'text'], dtype='str')


In [3]:
import pandas as pd

# 1️⃣ Read CSV
df = pd.read_csv("dataset/new.csv", encoding="latin1")

# 2️⃣ Rename column
df = df.rename(columns={"doc ID": "doc_id"})

# 3️⃣ Forward fill IDs and titles
df["doc_id"] = df["doc_id"].ffill()
df["title"] = df["title"].ffill()

# 4️⃣ SAFE text cleaning BEFORE groupby
df["text"] = df["text"].apply(
    lambda x: str(x) if pd.notna(x) else ""
)

# 5️⃣ SAFE groupby + join (FLOATS REMOVED HERE)
df_clean = (
    df.groupby(["doc_id", "title"], as_index=False)
      .agg({
          "text": lambda x: " ".join(
              [t for t in x if isinstance(t, str)]
          )
      })
)

# 6️⃣ Final check
print(df_clean.head())
print("Total documents:", len(df_clean))


   doc_id                    title  \
0     1.0  Artificial intelligence   
1     2.0         Machine learning   

                                                text  
0  Artificial intelligence (AI) is the capability...  
1  Machine learning (ML) is a field of study in a...  
Total documents: 2


In [4]:
# Add word count column
df_clean["word_count"] = df_clean["text"].apply(lambda x: len(str(x).split()))

# Stats
avg_length = df_clean["word_count"].mean()
max_length = df_clean["word_count"].max()

print("Average document length (words):", int(avg_length))
print("Maximum document length (words):", max_length)


Average document length (words): 550
Maximum document length (words): 699


In [5]:
from collections import Counter
import re

# Combine all text
all_text = " ".join(df_clean["text"].astype(str))

# Clean text (lowercase, remove symbols)
words = re.findall(r"\b[a-zA-Z]+\b", all_text.lower())

# Count words
word_freq = Counter(words)

# Top 20 most common words
print("Top 20 most used words:")
for word, count in word_freq.most_common(20):
    print(word, ":", count)


Top 20 most used words:
and : 52
learning : 34
the : 33
of : 33
to : 27
in : 25
a : 21
ai : 19
machine : 17
as : 14
with : 13
algorithms : 13
by : 11
that : 9
artificial : 8
intelligence : 8
is : 8
human : 8
field : 8
e : 7


In [6]:
from collections import Counter
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Combine all text
all_text = " ".join(df_clean["text"].astype(str)).lower()

# Extract words (only alphabets)
words = re.findall(r"\b[a-zA-Z]+\b", all_text)

# Filter: length >= 5 and remove stopwords
filtered_words = [
    w for w in words
    if len(w) >= 5 and w not in ENGLISH_STOP_WORDS
]

# Count frequency
word_freq = Counter(filtered_words)

# Top 20 most used words
print("Top 20 most used words (length ≥ 5):")
for word, count in word_freq.most_common(20):
    print(word, ":", count)


Top 20 most used words (length ≥ 5):
learning : 34
machine : 17
algorithms : 13
artificial : 8
intelligence : 8
human : 8
field : 8
computer : 6
neural : 6
tasks : 5
research : 5
methods : 5
mathematical : 5
networks : 5
machines : 4
goals : 4
generative : 4
cognitive : 4
study : 4
reinforcement : 4


In [ ]:
import pandas as pd
import json

# 1️⃣ Read CSV
df = pd.read_csv("dataset/new.csv", encoding="latin1")

# 2️⃣ Rename column (if needed)
df = df.rename(columns={"doc ID": "doc_id"})

# 3️⃣ Forward fill doc_id & title
df["doc_id"] = df["doc_id"].ffill()
df["title"] = df["title"].ffill()

# 4️⃣ Clean text safely
df["text"] = df["text"].apply(lambda x: str(x) if pd.notna(x) else "")

# 5️⃣ MERGE rows → ONE document
df_clean = (
    df.groupby(["doc_id", "title"], as_index=False)
      .agg({
          "text": lambda x: " ".join([t for t in x if isinstance(t, str)])
      })
)

# 6️⃣ Convert to JSON-safe dict
records = [
    {
        "doc_id": int(row["doc_id"]),
        "title": row["title"],
        "text": row["text"]
    }
    for _, row in df_clean.iterrows()
]

# 7️⃣ Save JSON (NO NaN, NO ERRORS)
with open("raw_docs.json", "w", encoding="utf-8") as f:
    json.dump(records, f, indent=2, ensure_ascii=False)

print("✅ raw_docs.json created correctly (document-level, no NaN)")
print("Total documents:", len(records))


✅ raw_docs.json created correctly (document-level, no NaN)
Total documents: 2
